## Dependencies

In [ ]:
import openpyxl #has to be imported before pandas
from openpyxl import load_workbook
import pandas as pd
import subprocess
import os
import urllib.request
import urllib.parse
from urllib.request import urlopen
import numpy as np
from urllib.request import urlopen
import requests
from pathlib import Path
from IPython.display import display # run jupyter Notebook as a python executable

## Generate_MC_to_MFdict

In [ ]:
def Generate_MC_to_MFdict():
    
    googlesheeturlprefix="https://docs.google.com/spreadsheets/export?id="
    MC_MF_lookup_sheet_ID = '1s2JbK4KlhTLrKRcrWD7ychPoN996pyqiKyFhqjfLJ9M'#
    googlesheeturlsuffix="&format=xlsx"
    MC_MF_lookupFile="MC_MF_lookup.xlsx"
    SHEET_NAME = 'Sheet1'
    url = googlesheeturlprefix+ MC_MF_lookup_sheet_ID + googlesheeturlsuffix
    
    #downlaod google sheet as xlsx file
    open(MC_MF_lookupFile, 'wb').write(requests.get(url).content) # actual download
    
    MC_2_MF_DF = pd.read_excel(MC_MF_lookupFile, sheet_name=SHEET_NAME)
    print('MC_2_MF_DF:')
    display(MC_2_MF_DF)
    
    MC_MFdict=MC_2_MF_DF.set_index('MC')['MF'].to_dict()
    print('MC_MFdict:')
    display(MC_MFdict)
    
    return(MC_MFdict)

## LoadWorking_MSAsampleetDF

In [ ]:
def LoadWorking_MSAsampleetDF():
    MSA_data_path = '/data'
    Working_MSAsampleetDF_path = MSA_data_path + "/" + "Working_MSAsampleetDF.csv"
    Working_MSAsampleetDF = pd.read_csv(Working_MSAsampleetDF_path)
    
    print('Working_MSAsampleetDF:')
    display(Working_MSAsampleetDF)
    
    return(Working_MSAsampleetDF)
    

## Download_IfP_MSA_expected_MC()

In [ ]:
def Download_IfP_MSA_expected_MC(MC_MFdictI):
    
    Google_SHEET_ID="1MepAtLnVNEO7foGta7fxKHaQ3AF35tlgC9Cg9OUTHng"
    filename = 'IfP_MSA_samples_expected_MC.xlsx'

    GOOGLE_SHEET_URL = f'https://docs.google.com/spreadsheets/d/{Google_SHEET_ID}/export?format=xlsx'

    try:
        response = requests.get(GOOGLE_SHEET_URL, timeout=10)
        response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)

        with open(filename, 'wb') as f:
            f.write(response.content)
        
        print(f"Download successful. File saved as '{filename}'.")
        
    except requests.exceptions.RequestException as e:
        print(f"Error during download or request: {e}")
        print("\nEnsure the Google Sheet is shared as 'Anyone with the link can view.'")
        return None
    
    except Exception as e:
        print(f"An error occurred during file processing: {e}")
        return None        

    # Read the expected MC of IfP MSA sample in a dataframe
    MethClasses=pd.read_excel(filename,sheet_name = 'MSA_samples_MC')
    
    MethClasses["Sample_Name"] = MethClasses["Sample_Name"].str.replace(".","_")
    
    #Methylation family for each Methylation class (MC) downloaded
    MethClasses['MF_int_Diag'] = MethClasses['MC_Int_Diag'].map(MC_MFdictI)
    
    print("MethClasses:")
    display(MethClasses)
    
    return MethClasses

## ParsePieCsv

In [ ]:
def ParsePieCsv(Working_MSAsamplSheetDFi, MC_MFdictI):
    
    #create a new columns for most abbundant methylation class
    Working_MSAsamplSheetDFi['MC01'] = "NA"
    
    #create a new columns for count of most abbundant methylation class
    Working_MSAsamplSheetDFi['CC01'] = "0"
    
    #create a new colum for the most abbundant methylation family
    Working_MSAsamplSheetDFi['MF01'] = "NA"
    
    #create a new colum for the casecount of the most abbundanat methlytion family
    Working_MSAsamplSheetDFi['MFCC01'] = "0"
    #
    for index, row in Working_MSAsamplSheetDFi.iterrows():
            
        ConcatenatedName = row['Sample_Name_InputM_Beta']
        
        print('ConcatenatedName:', {ConcatenatedName})
        
        PieCsvPath = "/data/nanodip_reports/" + ConcatenatedName +"_ND_IfP_20250912_pie.csv"
        CSV=pd.read_csv(PieCsvPath)
        
        print('CSV:')
        display(CSV)
        
        #pick most abbundant methylation class from pie.csv dataframe and return this value to column
        MC01=CSV.loc[0,"methClass"]
        print('MC01:',{MC01})
        Working_MSAsamplSheetDFi.loc[index,'MC01']=MC01
        
        #pick casese count of most abbundant methylation class from pie.csv dataframe
        CC01=CSV.loc[0,"caseCount"]
        print('CC01', {CC01})
        Working_MSAsamplSheetDFi.loc[index,'CC01']=CC01
        
        # derive MF and re-sort the resulting DF - including new summation of new casecounts      
        CSV['MF']=CSV['methClass'].map(MC_MFdictI)
        print('expanded CSV:')
        display(CSV)
        
        # create new dataframe from CSV by grouping by Methylationfamily(MF) and summarize the cc for idetical lines
        MF_grouped_DF = CSV.groupby('MF')['caseCount'].sum().reset_index()
        print('MF_grouped_DF:')
        display(MF_grouped_DF)
        
        Resorted_MF_grouped_DF = MF_grouped_DF.sort_values(by='caseCount', ascending=False)
        
        MF01=Resorted_MF_grouped_DF.loc[0,"MF"]
        Working_MSAsamplSheetDFi.loc[index,'MF01']=MF01
        
        MFCC01=Resorted_MF_grouped_DF.loc[0,"caseCount"]
        Working_MSAsamplSheetDFi['MFCC01']=MFCC01
    
    
    return(Working_MSAsamplSheetDFi)

## Expand_MSA_MC01_DFex_w_IntD_MC()

In [ ]:
def Expand_MSA_MC01_DFex_w_IntD_MC(MSA_MC01_DFi, IfP_IntDiag_MCi):
    DF_merged = pd.merge(
        left=MSA_MC01_DFi,
        right=IfP_IntDiag_MCi,
        on='Sample_Name', 
        how='left'
        )


    DF_merged['MC_eq_IntDiag'] = (DF_merged['MC01'] == DF_merged['MC_Int_Diag']).astype(int)
    DF_merged['MF_eq_IntDiag'] = (DF_merged['MF01'] == DF_merged['MF_int_Diag']).astype(int)
    
    #write DF_merged_as xlsx to HDD
    DF_merged_File_Path = "/data/" + "MSA_MC01_DF_w_IntD_MC.xlsx"
    DF_merged.to_excel(DF_merged_File_Path, index=False)
    
    return(DF_merged)

## main

In [ ]:
if __name__ == '__main__':
    Working_MSAsamplSheetDFex = LoadWorking_MSAsampleetDF()
    MC_MFdictEx = Generate_MC_to_MFdict()
    MSA_MC01_DFex = ParsePieCsv(Working_MSAsamplSheetDFex, MC_MFdictEx)# MC01  is the most abbundate methylation class determined in the UMAP analysis
    IfP_IntDiag_MCex = Download_IfP_MSA_expected_MC(MC_MFdictEx)
    Expand_MSA_MC01_DFex_w_Int_DF= Expand_MSA_MC01_DFex_w_IntD_MC(MSA_MC01_DFex, IfP_IntDiag_MCex)